PHASE 1: PRE-PROCESSING THE ORIGINAL DATASET

Loading the dataset

In [2]:
import pandas as pd

# Load the dataset
file_path = "Crimes_-_2001_to_Present.csv"  # Update this path
df = pd.read_csv(file_path)



Retrieving the details about Dataset

In [3]:
# 🔹 Display dataset info (column types, non-null counts)
print("🔹 Dataset Info:")
df.info()

# 🔹 Display first 5 rows to check format
print("\n🔹 Sample Data:")
print(df.head())

# 🔹 Check number of missing values per column
print("\n🔹 Missing Values Per Column:")
print(df.isnull().sum())

# 🔹 Check unique values per column (categorical overview)
print("\n🔹 Unique Values Per Column:")
print(df.nunique())

# 🔹 Summary statistics for numerical columns
print("\n🔹 Summary Statistics (Numerical Data):")
print(df.describe())

# 🔹 Check for duplicate rows
duplicate_count = df.duplicated().sum()
print(f"\n🔹 Number of Duplicate Rows: {duplicate_count}")

# 🔹 Check date range (earliest & latest crime records)
print("\n🔹 Date Range in Dataset:")
print(f"From {df['Date'].min()} to {df['Date'].max()}")

# 🔹 Check top 10 most frequent crime types
print("\n🔹 Top 10 Most Frequent Crime Types:")
print(df["Primary Type"].value_counts().head(10))

# 🔹 Check top 10 most affected districts
print("\n🔹 Top 10 Crime Affected Districts:")
print(df["District"].value_counts().head(10))

# 🔹 Save a small sample (first 100K rows) for further testing
df_sample = df.head(100000)
df_sample.to_csv("crime_data_sample_raw.csv", index=False)

print("\n✅ Analysis Complete! Upload 'crime_data_sample_raw.csv' for review.")


🔹 Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7846809 entries, 0 to 7846808
Data columns (total 22 columns):
 #   Column                Dtype  
---  ------                -----  
 0   ID                    int64  
 1   Case Number           object 
 2   Date                  object 
 3   Block                 object 
 4   IUCR                  object 
 5   Primary Type          object 
 6   Description           object 
 7   Location Description  object 
 8   Arrest                bool   
 9   Domestic              bool   
 10  Beat                  int64  
 11  District              float64
 12  Ward                  float64
 13  Community Area        float64
 14  FBI Code              object 
 15  X Coordinate          float64
 16  Y Coordinate          float64
 17  Year                  int64  
 18  Updated On            object 
 19  Latitude              float64
 20  Longitude             float64
 21  Location              object 
dtypes: bool(2), float64(7), in

Dropping Non Informative cols

In [4]:
# Drop unnecessary columns
drop_columns = ["ID", "Case Number", "Updated On", "Location", "X Coordinate", "Y Coordinate"]
df.drop(columns=drop_columns, inplace=True)

print(f"✅ Dropped columns: {drop_columns}")


✅ Dropped columns: ['ID', 'Case Number', 'Updated On', 'Location', 'X Coordinate', 'Y Coordinate']


Handling missing Latitude and Longitude

In [5]:
# Remove rows where both Latitude and Longitude are missing
df = df.dropna(subset=["Latitude", "Longitude"])

print(f"✅ Removed rows with missing Latitude/Longitude: {df.shape}")


✅ Removed rows with missing Latitude/Longitude: (7758698, 16)


Imputing Missing District, Ward, and Community Area

In [7]:
df["District"] = df["District"].ffill()
df["Ward"] = df["Ward"].ffill()
df["Community Area"] = df["Community Area"].ffill()

print(f"✅ Imputed missing values for District, Ward, Community Area")


✅ Imputed missing values for District, Ward, Community Area


Handling missing location descriptions

In [10]:
# Fill missing Location Description with "UNKNOWN"
df["Location Description"].fillna("UNKNOWN", inplace=True)

print("✅ Filled missing 'Location Description' with 'UNKNOWN'")


✅ Filled missing 'Location Description' with 'UNKNOWN'


Convert and Standardize Datatypes

In [11]:
# Convert Date column to datetime format
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y %I:%M:%S %p")

# Convert District, Ward, and Community Area to integers (after handling missing values)
df["District"] = df["District"].astype(int)
df["Ward"] = df["Ward"].astype(int)
df["Community Area"] = df["Community Area"].astype(int)

print("✅ Converted Date to datetime & standardized data types")


✅ Converted Date to datetime & standardized data types


Extract Temporal Features

In [12]:
# Extract year, month, day, hour, weekday, and season
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["Hour"] = df["Date"].dt.hour
df["DayOfWeek"] = df["Date"].dt.dayofweek  # Monday = 0, Sunday = 6

# Define seasons based on months
season_map = {12: "Winter", 1: "Winter", 2: "Winter",
              3: "Spring", 4: "Spring", 5: "Spring",
              6: "Summer", 7: "Summer", 8: "Summer",
              9: "Fall", 10: "Fall", 11: "Fall"}

df["Season"] = df["Month"].map(season_map)

print("✅ Extracted temporal features")


✅ Extracted temporal features


Removing Outliers

In [13]:
# Define valid coordinate range for Chicago
valid_latitude_range = (41.6, 42.1)
valid_longitude_range = (-87.9, -87.5)

# Filter out rows with invalid coordinates
df = df[(df["Latitude"].between(*valid_latitude_range)) & 
        (df["Longitude"].between(*valid_longitude_range))]

print(f"✅ Removed outliers based on Latitude/Longitude: {df.shape}")


✅ Removed outliers based on Latitude/Longitude: (7732599, 21)


Keeping only essential cols

In [14]:
# Keep only relevant columns
keep_columns = ["Date", "Year", "Month", "Day", "Hour", "DayOfWeek", "Season", 
                "Primary Type", "District", "Community Area", "Ward", 
                "Latitude", "Longitude", "Arrest", "Domestic"]

df = df[keep_columns]

print(f"✅ Kept essential columns: {keep_columns}")


✅ Kept essential columns: ['Date', 'Year', 'Month', 'Day', 'Hour', 'DayOfWeek', 'Season', 'Primary Type', 'District', 'Community Area', 'Ward', 'Latitude', 'Longitude', 'Arrest', 'Domestic']


Saving the cleaned Dataset

In [15]:
# Save cleaned dataset for further processing
df.to_csv("cleaned_crime_data.csv", index=False)

print(f"✅ Saved cleaned dataset with {df.shape[0]} rows and {df.shape[1]} columns")


✅ Saved cleaned dataset with 7732599 rows and 15 columns
